# MIMIC-IV General Item Exploration

Inspect the lab, chart, and ICU input item mappings used by `MIMICIVMultiTargetAdapter` before retrieving patient-level data. This notebook reads only MIMIC dictionary tables (`hosp/d_labitems.csv.gz` and `icu/d_items.csv.gz`) plus the local adapter config.

## Setup

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'src' / 'interpretable_ts_vit').exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError(f'Could not find repository root from {cwd}')

SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(PROJECT_DIR)

CONFIG_PATH = PROJECT_DIR / 'configs' / 'mimic_targets.yaml'
print('PROJECT_DIR:', PROJECT_DIR)
print('CONFIG_PATH:', CONFIG_PATH)

In [ ]:
import pandas as pd

from interpretable_ts_vit.datasets import load_mimic_targets_config
from interpretable_ts_vit.datasets.mimic_iv import _MIMICSource
from interpretable_ts_vit.datasets.mimic_targets import _match_patterns

config = load_mimic_targets_config(CONFIG_PATH)
mimic_path = PROJECT_DIR / config.mimic_path if not Path(config.mimic_path).is_absolute() else Path(config.mimic_path)
cache_dir = PROJECT_DIR / config.cache_dir if config.cache_dir and not Path(config.cache_dir).is_absolute() else Path(config.cache_dir) if config.cache_dir else None
extraction_dir = cache_dir / 'dictionary_extracted' if cache_dir and config.use_extracted_files else None
source = _MIMICSource(mimic_path, extraction_dir=extraction_dir)

print('MIMIC source:', mimic_path)
print('Dictionary extraction dir:', extraction_dir)
print('Patient-level tables are not read in this notebook.')

## Read Dictionary Tables Only

In [ ]:
lab_items = source.read_table('hosp/d_labitems.csv.gz', usecols=['itemid', 'label', 'fluid', 'category'])
icu_items = source.read_table('icu/d_items.csv.gz', usecols=['itemid', 'label', 'abbreviation', 'category', 'unitname', 'param_type'])

lab_items['search_text'] = lab_items[['label', 'fluid', 'category']].fillna('').astype(str).agg(' '.join, axis=1).str.lower()
icu_items['search_text'] = icu_items[['label', 'abbreviation', 'category', 'unitname', 'param_type']].fillna('').astype(str).agg(' '.join, axis=1).str.lower()

print('d_labitems rows:', len(lab_items))
print('d_items rows:', len(icu_items))
display(lab_items.head())
display(icu_items.head())

## Lab Item Regex Matches

These are the `hosp/labevents.csv.gz` item IDs that would be selected by the configured `lab_regexes` plus any exact `lab_itemids` overrides.

In [ ]:
lab_match_rows = []
for variable, patterns in config.lab_regexes.items():
    matched = lab_items[_match_patterns(lab_items['search_text'], patterns)].copy()
    exact_ids = {int(itemid) for itemid in config.lab_itemids.get(variable, [])}
    if exact_ids:
        matched = pd.concat([matched, lab_items[lab_items['itemid'].isin(exact_ids)]], ignore_index=True).drop_duplicates('itemid')
    for _, row in matched.sort_values(['label', 'itemid']).iterrows():
        lab_match_rows.append({
            'variable': variable,
            'itemid': int(row['itemid']),
            'label': row['label'],
            'fluid': row['fluid'],
            'category': row['category'],
            'patterns': ', '.join(patterns),
            'exact_override': int(row['itemid']) in exact_ids,
        })

lab_matches = pd.DataFrame(lab_match_rows)
display(lab_matches)
display(lab_matches.groupby('variable')['itemid'].nunique().rename('matched_lab_itemids').to_frame() if not lab_matches.empty else pd.DataFrame())

## Configured Chart Item IDs

These are exact `icu/chartevents.csv.gz` item IDs from `chart_itemids`, resolved against `icu/d_items.csv.gz`.

In [ ]:
chart_rows = []
for variable, itemids in config.chart_itemids.items():
    matched = icu_items[icu_items['itemid'].isin([int(itemid) for itemid in itemids])].copy()
    missing = sorted(set(int(itemid) for itemid in itemids) - set(matched['itemid'].astype(int)))
    for _, row in matched.sort_values('itemid').iterrows():
        chart_rows.append({
            'variable': variable,
            'itemid': int(row['itemid']),
            'label': row['label'],
            'abbreviation': row.get('abbreviation'),
            'category': row.get('category'),
            'unitname': row.get('unitname'),
            'param_type': row.get('param_type'),
            'missing_from_dictionary': False,
        })
    for itemid in missing:
        chart_rows.append({
            'variable': variable,
            'itemid': itemid,
            'label': None,
            'abbreviation': None,
            'category': None,
            'unitname': None,
            'param_type': None,
            'missing_from_dictionary': True,
        })

chart_matches = pd.DataFrame(chart_rows)
display(chart_matches)
display(chart_matches.groupby('variable')['itemid'].nunique().rename('configured_chart_itemids').to_frame() if not chart_matches.empty else pd.DataFrame())

## ICU Input Item Regex Matches

These are `icu/inputevents.csv.gz` item IDs selected from `icu/d_items.csv.gz` by `inputevent_regexes`.

In [ ]:
input_rows = []
for variable, patterns in config.inputevent_regexes.items():
    matched = icu_items[_match_patterns(icu_items['search_text'], patterns)].copy()
    for _, row in matched.sort_values(['label', 'itemid']).iterrows():
        input_rows.append({
            'variable': variable,
            'itemid': int(row['itemid']),
            'label': row['label'],
            'abbreviation': row.get('abbreviation'),
            'category': row.get('category'),
            'unitname': row.get('unitname'),
            'param_type': row.get('param_type'),
            'patterns': ', '.join(patterns),
        })

input_matches = pd.DataFrame(input_rows)
display(input_matches)
display(input_matches.groupby('variable')['itemid'].nunique().rename('matched_input_itemids').to_frame() if not input_matches.empty else pd.DataFrame())

## Medication Drug Regexes

`hosp/prescriptions.csv.gz` contains patient-level drug rows, so this notebook does not read it. Review these regexes before running data creation; after generation, the target dataset exploration notebook can show observed medication coverage.

In [ ]:
drug_patterns = pd.DataFrame(
    [{'variable': variable, 'pattern': pattern} for variable, patterns in config.drug_regexes.items() for pattern in patterns]
)
display(drug_patterns)
display(drug_patterns.groupby('variable')['pattern'].count().rename('n_patterns').to_frame())

## Quick Search Helpers

Use these cells to manually inspect candidate dictionary rows before editing `configs/mimic_targets.yaml`.

In [ ]:
LAB_SEARCH = 'troponin'
display(lab_items[lab_items['search_text'].str.contains(LAB_SEARCH, case=False, regex=False, na=False)].drop(columns=['search_text']).head(50))

In [ ]:
ICU_SEARCH = 'dextrose'
display(icu_items[icu_items['search_text'].str.contains(ICU_SEARCH, case=False, regex=False, na=False)].drop(columns=['search_text']).head(50))